# Annotate via LangGraph

In [2]:
import { readFileSync } from 'node:fs';
import { START, END, StateGraph } from "@langchain/langgraph";
import { Annotations } from "@common/annotations.ts";
import { getLogger } from '../../server/src/Logger.ts';
import { StateInfo, responseContent } from '../../server/src/agents/agent.ts';
import { encodeAnnotationsNode } from "../../server/src/agents/nodes/encodeAnnotationsNode.ts";
import { modelNode } from "../../server/src/agents/nodes/modelNode.ts";
import { HumanMessage, SystemMessage } from "@langchain/core/messages";

export const annNode = (state: typeof StateInfo.State) => {
  const PROMPT = readFileSync('./annotateNodePromptCategories.txt', 'utf-8');
  state.logger.info(`System Message:\n${PROMPT}`);
  state.logger.info(`Human Message:\n${state.systemData}`);
  return { messages: [
    new SystemMessage(PROMPT),
    new HumanMessage(state.systemData)
  ]};
}

// Define a new graph
const workflow = new StateGraph(StateInfo)
  .addNode("encodeAnnotations", encodeAnnotationsNode)
  .addNode("annotate", annNode)
  .addNode("model", modelNode)
  .addEdge(START, "encodeAnnotations")
  .addEdge("encodeAnnotations","annotate")
  .addEdge("annotate", "model")
  .addEdge("model", END);

// Compile graph
export const graph = workflow.compile();

const logger = getLogger();
const userUUID: string = "0";
const lhsText = readFileSync('../../frontend/public/data/AES/selected-text.txt', 'utf-8');
const rhsText = readFileSync('../../frontend/public/data/AES/pre-written.txt', 'utf-8');
const currentAnnotations: Annotations = { "mappings": [], "lhsLabels": [], "rhsLabels": [] };

const config = { configurable: { thread_id: userUUID } };
const output = await graph.invoke({ lhsText: lhsText, rhsText: rhsText, currentAnnotations: currentAnnotations, logger: logger }, config);
console.log(responseContent(output));


### CHAIN-OF-THOUGHT ANALYSIS

I'll analyze the LHS TEXT and identify semantically contiguous segments, providing appropriate labels for each.

1. The first segment introduces the general function for AES and its inverse, explaining the core concepts of rounds and round keys. This is an overview of the algorithm.

2. The next segment explains key expansion and how the three AES variants differ, including a table showing key-block-round combinations.

3. The text then defines the inputs to the CIPHER() function and shows how AES-128, AES-192, and AES-256 are defined in terms of CIPHER().

4. Section 5.1 introduces the four transformations used in CIPHER(): SubBytes(), ShiftRows(), MixColumns(), and AddRoundKey().

5. Algorithm 1 provides the pseudocode for CIPHER(), followed by an explanation of the steps.

6. Sections 5.1.1 through 5.1.4 provide detailed descriptions of each transformation.

7. Section 5.3 introduces INVCIPHER() and its components.

8. Section 5.3.3 specifically detail